In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

print("Libraries imported")

Libraries imported


In [15]:
train = pd.read_csv("../data/train.csv")

train["datetime"] = pd.to_datetime(train["datetime"])

train["year"] = train["datetime"].dt.year
train["month"] = train["datetime"].dt.month
train["day"] = train["datetime"].dt.day
train["hour"] = train["datetime"].dt.hour
train["day_of_week"] = train["datetime"].dt.dayofweek

train["is_weekend"] = (
    train["day_of_week"] >= 5
).astype(int)

train["peak_hour"] = (
    (
        (train["day_of_week"] < 5) &
        (train["hour"].isin([7, 8, 9, 17, 18, 19]))
    )
    |
    (
        (train["day_of_week"] >= 5) &
        (train["hour"].between(10, 18))
    )
).astype(int)

print("Feature engineering completed")

Feature engineering completed


In [16]:
y = train["count"]

X = train.drop(
    columns=["count", "casual", "registered", "datetime"]
)

X = pd.get_dummies(
    X,
    columns=["season", "weather"],
    drop_first=False
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training:", X_train.shape)
print("Validation:", X_valid.shape)

Training: (8708, 21)
Validation: (2178, 21)


In [17]:
from sklearn.metrics import mean_squared_log_error

def rmsle(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(
        mean_squared_log_error(y_true, y_pred)
    )

In [18]:
from sklearn.ensemble import RandomForestRegressor

baseline_rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

baseline_rf.fit(X_train, y_train)

baseline_pred = baseline_rf.predict(X_valid)

baseline_score = rmsle(
    y_valid,
    baseline_pred
)

print("Baseline Random Forest RMSLE:", baseline_score)

Baseline Random Forest RMSLE: 0.3281389526975628


In [19]:
rf_300 = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_300.fit(X_train, y_train)

pred_300 = rf_300.predict(X_valid)

score_300 = rmsle(
    y_valid,
    pred_300
)

print("Random Forest 300 trees RMSLE:", score_300)

Random Forest 300 trees RMSLE: 0.32750459396147674


In [20]:
depth_results = []

for depth in [15, 25, None]:

    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=depth,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_valid)

    score = rmsle(y_valid, pred)

    depth_results.append({
        "max_depth": depth,
        "RMSLE": score
    })

depth_results = pd.DataFrame(depth_results)

depth_results

,max_depth,RMSLE
0,15.0,0.327600
1,25.0,0.328351
2,NaN,0.328139


In [21]:
leaf_results = []

for min_leaf in [1, 2, 4]:

    model = RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=min_leaf,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_valid)

    score = rmsle(y_valid, pred)

    leaf_results.append({
        "min_samples_leaf": min_leaf,
        "RMSLE": score
    })

leaf_results = pd.DataFrame(leaf_results)

leaf_results

,min_samples_leaf,RMSLE
0,1,0.328139
1,2,0.327041
2,4,0.329890


In [22]:
tuning_results = pd.concat([
    pd.DataFrame([{
        "Experiment": "Baseline",
        "Parameters": "200 trees",
        "RMSLE": baseline_score
    }]),
    
    pd.DataFrame([{
        "Experiment": "More trees",
        "Parameters": "300 trees",
        "RMSLE": score_300
    }])
], ignore_index=True)

tuning_results

,Experiment,Parameters,RMSLE
0,Baseline,200 trees,0.328139
1,More trees,300 trees,0.327505


In [23]:
from sklearn.ensemble import GradientBoostingRegressor

gb_results = []

configs = [
    (200, 0.05, 5),
    (300, 0.05, 5),
    (300, 0.03, 6)
]

for n_estimators, learning_rate, max_depth in configs:

    model = GradientBoostingRegressor(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        random_state=42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_valid)

    score = rmsle(y_valid, pred)

    gb_results.append({
        "n_estimators": n_estimators,
        "learning_rate": learning_rate,
        "max_depth": max_depth,
        "RMSLE": score
    })

gb_results = pd.DataFrame(gb_results)

gb_results

,n_estimators,learning_rate,max_depth,RMSLE
0,200,0.05,5,0.452745
1,300,0.05,5,0.459516
2,300,0.03,6,0.397612


In [24]:
best_gb = gb_results.loc[
    gb_results["RMSLE"].idxmin()
]

best_gb

n_estimators     300.000000
learning_rate      0.030000
max_depth          6.000000
RMSLE              0.397612
Name: 2, dtype: float64

In [25]:
final_tuning_results = pd.DataFrame({
    "Model": [
        "Baseline Random Forest",
        "Random Forest - 300 trees",
        "Best Gradient Boosting"
    ],
    "RMSLE": [
        baseline_score,
        score_300,
        best_gb["RMSLE"]
    ]
})

final_tuning_results = final_tuning_results.sort_values(
    "RMSLE"
).reset_index(drop=True)

final_tuning_results

,Model,RMSLE
0,Random Forest - 300 trees,0.327505
1,Baseline Random Forest,0.328139
2,Best Gradient Boosting,0.397612


In [26]:
import os

os.makedirs("../results", exist_ok=True)

final_tuning_results.to_csv(
    "../results/tuning_results.csv",
    index=False
)

print("Tuning results saved.")

Tuning results saved.
